In [ ]:
# Cette cellule s'exécute uniquement si on est sur Google Colab
import sys

if "google.colab" in sys.modules:
    # 1. On clone ton projet GitHub (mettez à jour la branche si nécessaire)
    !git clone -b team/NN/Yann https://github.com/ST4-BlackSwan/Higgs-TeamB.git

    # 2. On se déplace mathématiquement dans le dossier qui contient main.py et data.py
    import os

    os.chdir("Higgs-TeamB")

    # 3. On ajoute le dossier au chemin de recherche Python
    sys.path.append(os.getcwd())
    print("Environnement Colab configuré avec succès !")

In [1]:
%pip install HiggsML==0.1.4
#penser à restart le kernel après avoir fait ça

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 6.7 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/ST4-BlackSwan/Higgs-TeamB.git

Cloning into 'Higgs-TeamB'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 480 (delta 87), reused 76 (delta 49), pack-reused 324 (from 2)
Receiving objects: 100% (480/480), 10.35 MiB | 21.49 MiB/s, done.
Resolving deltas: 100% (240/240), done.


In [3]:
!ls

Higgs-TeamB  sample_data


In [4]:
!pwd
%cd Higgs-TeamB/
!pwd
!git checkout team/NN/main

/content
/content/Higgs-TeamB
/content/Higgs-TeamB
Branch 'team/NN/main' set up to track remote branch 'team/NN/main' from 'origin'.
Switched to a new branch 'team/NN/main'


In [5]:
!git pull

Already up to date.


In [6]:
!ls NN

data.py  distribution.py  main.py  README.md  significance.py  weighted_roc.py


In [7]:
!git status

On branch team/NN/main
Your branch is up to date with 'origin/team/NN/main'.

nothing to commit, working tree clean


In [8]:
!python NN/main.py

2026-06-03 11:28:02.865759: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-03 11:28:02.932024: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [24]:
from NN.data import load_and_clean_blackswan, prepare_datasets

print("1/2 - Téléchargement et nettoyage des données via data.py...")
features, target, weights = load_and_clean_blackswan()

print("\n2/2 - Découpage, standardisation (StandardScaler) et équilibrage des poids...")
X_train, X_test, y_train, y_test, w_train, w_test, scaler = prepare_datasets(features, target, weights)

print("\n✅ Traitement terminé avec succès !")
print(f"Dimensions Train : {X_train.shape} | Dimensions Test : {X_test.shape}")

In [25]:
from NN.main import NeuralNetwork

print("Initialisation du modèle séquentiel Keras...")
nn = NeuralNetwork(X_train)

print("\nDémarrage de l'entraînement des époques (Optimiseur Adam)...")
nn.fit(X_train, y_train, weights_train=w_train)

print("\n✅ Réseau de neurones entraîné !")

In [9]:
import cv2
import matplotlib.pyplot as plt
from NN.weighted_roc import calculate_weighted_roc
from NN.distribution import plot_score_separability
from NN.significance import optimize_poisson_significance

print("1/3 - Calcul des prédictions sur l'ensemble de test...")
predictions = nn.predict(X_test)

print("\n2/3 - Génération des métriques et des graphiques physiques...")
calculate_weighted_roc(y_test, predictions, sample_weights=w_test)
plot_score_separability(y_test, predictions, sample_weights=w_test)

print("\n3/3 - Calcul du seuil de sélection optimal...")
optimal_cut, max_sig = optimize_poisson_significance(y_test, predictions, sample_weights=w_test)

# --- Affichage des diagnostics générés ---
print("\n📊 TABLEAU DE BORD DES PERFORMANCES :")
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

img_roc = cv2.imread("nn_day1_roc_curve.png")
axes[0].imshow(cv2.cvtColor(img_roc, cv2.COLOR_BGR2RGB))
axes[0].axis('off')

img_dist = cv2.imread("nn_day1_score_distribution.png")
axes[1].imshow(cv2.cvtColor(img_dist, cv2.COLOR_BGR2RGB))
axes[1].axis('off')

img_sig = cv2.imread("nn_day1_significance_scan.png")
axes[2].imshow(cv2.cvtColor(img_sig, cv2.COLOR_BGR2RGB))
axes[2].axis('off')

plt.tight_layout()
plt.show()

1/3 - Calcul des prédictions sur l'ensemble de test...


/content/Higgs-TeamB/NN/significance.py:65: SyntaxWarning: invalid escape sequence '\s'
  plt.title(f"Significance Scan Profile (Max: {max_significance:.3f} $\sigma$)")


NameError: name 'nn' is not defined